In [ ]:
import os
import sys
import time
import numpy as np
import scipy
import jax
import jax.numpy as jnp
import viser

sys.path.append(os.path.abspath(".."))
from environments import ClothEnv

In [ ]:
server = viser.ViserServer()
_ = server.scene.add_grid(name="ground")

In [ ]:
params = {
    'time_step': 0.02,       # s
    'thickness': 1e-3,       # m
    'youngs_modulus': 1e5,  # Pa
    'possion_ratio': 0.3,
    'mass_density': 200,     # kg/m³
}
env = ClothEnv.from_regular_grid(cloth_width=0.3, num_div_side=9, **params)
env = ClothEnv.from_obj_file("assets/shirt.obj", **params)

# Specify initial node positions
R = scipy.spatial.transform.Rotation.from_rotvec([np.deg2rad(-30), np.deg2rad(60), 0])
R = jnp.array(R.as_matrix())
x_node = env.params.x_node_rest @ R
x_node = x_node.at[:, 2].add(0.3)

state = env.state(x_node=x_node)
control = env.control()
env.visualize(server, state)

In [ ]:
for i in range(99999):
    if i % 250 == 0:
        state = env.state(x_node=x_node)

    start = time.time()
    state, lin = jax.vjp(env.step, state, control)
    A, B = jax.vmap(lin)(jnp.eye(state.size))
    elapsed = time.time() - start
    print(f"\rStep took {elapsed*1e3:.2f} ms  ", end="")

    if 'A' in locals() and jnp.isnan(A).any():
        print(A)
        raise RuntimeError("NaN occurred")
    if 'B' in locals() and jnp.isnan(B).any():
        print(B)
        raise RuntimeError("NaN occurred")

    env.visualize(server, state)

    wait = env.params.dt - elapsed
    if wait > 0:
        time.sleep(wait)